**Step 1: Load Data into Pandas**

In [1]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')



# Load data
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Uber Supply Demand/Uber_Supply_Demand_Cleaned.csv')


df.head()

Mounted at /content/drive


,Request_id,Pickup_point,Driver_id,Status,Request_timestamp,Drop_timestamp,Date,Hour,Weekday,Time_Slot,Driver_Assigned,Trip_Duration_Minutes
0,619,Airport,1.0,Trip Completed,2016-07-11 11:51:00,2016-07-11 13:00:00,2016-07-11,11,Monday,Morning,Yes,69.000000
1,867,Airport,1.0,Trip Completed,2016-07-11 17:57:00,2016-07-11 18:47:00,2016-07-11,17,Monday,Evening,Yes,50.000000
2,1807,City,1.0,Trip Completed,2016-07-12 09:17:00,2016-07-12 09:58:00,2016-07-12,9,Tuesday,Morning,Yes,41.000000
3,2532,Airport,1.0,Trip Completed,2016-07-12 21:08:00,2016-07-12 22:03:00,2016-07-12,21,Tuesday,Evening,Yes,55.000000
4,3112,City,1.0,Trip Completed,2016-07-13 08:33:16,2016-07-13 09:25:47,2016-07-13,8,Wednesday,Morning,Yes,52.516667


**Step 2: Create SQLite Database**

In [2]:
import sqlite3

conn = sqlite3.connect('uber.db')

df.to_sql(
    'uber',
    conn,
    if_exists='replace',
    index=False
)

print("Table Created Successfully")

Table Created Successfully


**Step 3: Helper Function for SQL Queries**

In [3]:
def run_query(query):
    return pd.read_sql_query(query, conn)

**SQL Analysis Queries**
**1. Total Ride Requests**

In [4]:
query = """
SELECT COUNT(*) AS Total_Requests
FROM uber;
"""

run_query(query)

,Total_Requests
0,6745


**2. Trip Status Distribution**

In [5]:
query = """
SELECT Status,
       COUNT(*) AS Total
FROM uber
GROUP BY Status
ORDER BY Total DESC;
"""

run_query(query)

,Status,Total
0,Trip Completed,2831
1,No Cars Available,2650
2,Cancelled,1264


**3. Requests by Pickup Point**

In [6]:
query = """
SELECT Pickup_point,
       COUNT(*) AS Requests
FROM uber
GROUP BY Pickup_point;
"""

run_query(query)

,Pickup_point,Requests
0,Airport,3238
1,City,3507


**4. Status by Pickup Point**

In [7]:
query = """
SELECT Pickup_point,
       Status,
       COUNT(*) AS Total
FROM uber
GROUP BY Pickup_point, Status
ORDER BY Pickup_point;
"""

run_query(query)

,Pickup_point,Status,Total
0,Airport,Cancelled,198
1,Airport,No Cars Available,1713
2,Airport,Trip Completed,1327
3,City,Cancelled,1066
4,City,No Cars Available,937
5,City,Trip Completed,1504


***Insight:***

**Airport:**

High No Cars Available

**City:**

High Cancellation Rate

**5. Peak Demand Hours**

In [8]:
query = """
SELECT Hour,
       COUNT(*) AS Requests
FROM uber
GROUP BY Hour
ORDER BY Requests DESC;
"""

run_query(query)

,Hour,Requests
0,18,510
1,20,492
2,19,473
3,21,449
4,5,445
5,9,431
6,8,423
7,17,418
8,7,406
9,6,398


**6. Time Slot Analysis**

In [9]:
query = """
SELECT Time_Slot,
       COUNT(*) AS Requests
FROM uber
GROUP BY Time_Slot
ORDER BY Requests DESC;
"""

run_query(query)

,Time_Slot,Requests
0,Morning,2517
1,Evening,2342
2,Night,1076
3,Afternoon,810


**Expected:**

Time Slot	Requests

Morning	Highest

Evening	Second Highest

**7. Driver Assignment Analysis**

In [10]:
query = """
SELECT Driver_Assigned,
       COUNT(*) AS Requests
FROM uber
GROUP BY Driver_Assigned;
"""

run_query(query)

,Driver_Assigned,Requests
0,No,2650
1,Yes,4095


Insight

Shows proportion of requests receiving drivers.

**8. Average Trip Duration**

In [11]:
query = """
SELECT ROUND(
AVG(Trip_Duration_Minutes),2
) AS Avg_Trip_Duration
FROM uber
WHERE Status='Trip Completed';
"""

run_query(query)

,Avg_Trip_Duration
0,52.41


**9. Top 10 Busiest Hours**

In [12]:
query = """
SELECT Hour,
       COUNT(*) AS Requests
FROM uber
GROUP BY Hour
ORDER BY Requests DESC
LIMIT 10;
"""
run_query(query)

,Hour,Requests
0,18,510
1,20,492
2,19,473
3,21,449
4,5,445
5,9,431
6,8,423
7,17,418
8,7,406
9,6,398


**10. Demand-Supply Gap**

In [13]:
query = """
SELECT
COUNT(*) AS Total_Requests,

SUM(
CASE
WHEN Status='Trip Completed'
THEN 1
ELSE 0
END
) AS Completed,

SUM(
CASE
WHEN Status!='Trip Completed'
THEN 1
ELSE 0
END
) AS Unfulfilled
FROM uber;
"""

run_query(query)

,Total_Requests,Completed,Unfulfilled
0,6745,2831,3914


**11. Fulfillment Rate**

In [14]:
query = """
SELECT
ROUND(
100.0 *
SUM(CASE WHEN Status='Trip Completed'
THEN 1 ELSE 0 END)
/ COUNT(*),
2
) AS Fulfillment_Rate
FROM uber;
"""

run_query(query)

,Fulfillment_Rate
0,41.97


Meaning nearly 58% demand is unmet.

**12. Airport Demand Analysis**

In [15]:
query = """
SELECT Status,
       COUNT(*) AS Total
FROM uber
WHERE Pickup_point='Airport'
GROUP BY Status;
"""

run_query(query)

,Status,Total
0,Cancelled,198
1,No Cars Available,1713
2,Trip Completed,1327


**Insight**

Airport experiences the highest number of:

No Cars Available

**13. City Demand Analysis**

In [16]:
query = """
SELECT Status,
       COUNT(*) AS Total
FROM uber
WHERE Pickup_point='City'
GROUP BY Status;
"""

run_query(query)

,Status,Total
0,Cancelled,1066
1,No Cars Available,937
2,Trip Completed,1504


**Insight**

City experiences:

High driver cancellations

**14. Demand by Weekday**

In [17]:
query = """
SELECT Weekday,
       COUNT(*) AS Requests
FROM uber
GROUP BY Weekday
ORDER BY Requests DESC;
"""

run_query(query)

,Weekday,Requests
0,Friday,1381
1,Monday,1367
2,Thursday,1353
3,Wednesday,1337
4,Tuesday,1307


**15. Average Trip Duration by Pickup Point**

In [18]:
query = """
SELECT Pickup_point,
       ROUND(
       AVG(Trip_Duration_Minutes),2
       ) AS Avg_Duration
FROM uber
WHERE Status='Trip Completed'
GROUP BY Pickup_point;
"""

run_query(query)

,Pickup_point,Avg_Duration
0,Airport,52.24
1,City,52.57


Final Business Conclusions
**Problem 1: Airport → City**

No Cars Available = Very High

Drivers are insufficient near the airport.

**Problem 2: City → Airport**

Cancellation Rate = Very High

Drivers may avoid long airport trips.

**Problem 3: Morning & Evening Rush Hours**

Demand > Supply

Peak-hour shortages cause lost revenue.

**Recommendations**

Increase airport driver incentives.

Introduce peak-hour surge pricing.

Predict demand using historical data.

Improve driver allocation algorithms.

Reward drivers for airport trips.